# Is `MIX_TN > 10,000` the right "Large Volume" cut-off?  *(Python)*
Runnable twin of `scripts/2026-06-15_large_project_threshold_analysis.py`. Derives where "large" actually begins from CSI's asphalt-bid tonnage, using the EDA-6411 toolkit (distributions, anomaly fences, K-means+silhouette, GMM/BIC). **Reads derived CSVs only; the live 10k rule is unchanged.**

In [ ]:
from pathlib import Path
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 50)

# --- absolute paths (edit BASE if you move the project) ----------------------
BASE = Path("/Users/t/Library/Mobile Documents/com~apple~CloudDocs/00 MSBA + Coding/"
            "00_summer 2026/00_CSI_ELP/01_Summer_2026")
OUT  = BASE / "02_modules/sweet_spot/outputs"
base = pd.read_csv(OUT / "2026-06-04_project_base.csv")
arch = pd.read_csv(OUT / "2026-06-04_project_archetypes.csv")[["Contract_Id", "rule_archetype"]]
df = base.merge(arch, on="Contract_Id", how="left")
print("merged:", df.shape, "| archetypes:", df["rule_archetype"].notna().sum())

In [ ]:
# primary series: CSI's asphalt bids (MIX_TN > 0); overlay: whole market w/ asphalt
has_mix = df["MIX_TN_num"] > 0
bid = df[has_mix & (df["CSI_Participated"].astype(str).str.upper() == "TRUE")].copy()
m = bid["MIX_TN_num"].astype(float)
m_all = df.loc[has_mix, "MIX_TN_num"].astype(float)
logm = np.log10(m.values)
CURRENT = 10_000
print(f"asphalt bids: {len(m)}  |  market w/ asphalt: {len(m_all)}")

### (1) Distribution & skew
Tonnage is heavily right-skewed → log scale. Where does 10k fall?

In [ ]:
print(m.describe().round(0).to_string())
from scipy import stats
print("skew raw =", round(stats.skew(m),2), " skew log =", round(stats.skew(logm),2))
pcts = {p: round(float(np.percentile(m,p))) for p in [10,25,50,75,90,95,99]}
print("percentiles:", pcts)
print(f">>> 10k = {(m<CURRENT).mean()*100:.0f}th pctile; {(m>CURRENT).mean()*100:.0f}% of bids exceed it")

### (2) Log-normality (Anderson–Darling on log10)

In [ ]:
ad = stats.anderson(logm, dist="norm")
print(f"A^2={ad.statistic:.2f}  5% crit={dict(zip(ad.significance_level, ad.critical_values)).get(5.0):.2f}")
print("log is far less skewed than raw -> do break-finding on log scale")

### (3) Upper-tail fences (anomaly detection: 1.5×IQR, mean+3σ)

In [ ]:
q1,q3 = np.percentile(m,[25,75]); iqr=q3-q1
iqr_fence = q3 + 1.5*iqr
sigma3_log = 10**(logm.mean()+3*logm.std())
print(f"1.5xIQR upper fence: {iqr_fence:,.0f}   mean+3sigma(log): {sigma3_log:,.0f}")
print("(sigma fences are huge/degenerate on a heavy tail -> a useful negative result)")

### (4) Natural breaks — K-means + silhouette, then GMM by BIC

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
X = logm.reshape(-1,1); Xs = StandardScaler().fit_transform(X)
best=(None,-1,None)
for k in range(2,6):
    km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(Xs)
    s = silhouette_score(Xs, km.labels_)
    print(f"  k={k}: silhouette={s:.3f}")
    if s>best[1]: best=(k,s,km)
k,sil,km = best
sc = StandardScaler().fit(X)
centers = np.sort(10**sc.inverse_transform(km.cluster_centers_).ravel())
kmeans_break = 10**((np.log10(centers[-1])+np.log10(centers[-2]))/2)
print(f"-> k={k}; centers(TN)={[round(c) for c in centers]}; top-two break={kmeans_break:,.0f}")

In [ ]:
from sklearn.mixture import GaussianMixture
best_g=(None,np.inf,None)
for n in range(1,5):
    g = GaussianMixture(n_components=n, random_state=42, n_init=5).fit(X)
    print(f"  n={n}: BIC={g.bic(X):.0f}")
    if g.bic(X)<best_g[1]: best_g=(n,g.bic(X),g)
n,_,g = best_g
grid = np.linspace(logm.min(), logm.max(), 4000).reshape(-1,1)
hi = int(np.argmax(g.means_.ravel())); sw = np.where(g.predict(grid)==hi)[0]
gmm_cross = 10**grid[sw[0],0] if (n>=2 and len(sw)) else np.nan
print(f"-> {n} components; GMM crossover into top mode = {gmm_cross:,.0f} (≈10k validates the floor)")

### (5) Does behavior shift at 10k? Outcome by tonnage band
Win-well = Excellent/Acceptable Win; Bomb = won but overpaid. Conditional on bidding.

In [ ]:
WIN_WELL = {"Excellent Win","Acceptable Win"}
bid["is_win_well"] = bid["CSI_Bid_Quality"].isin(WIN_WELL)
bid["is_bomb"] = bid["CSI_Bid_Quality"]=="Bomb"
edges=[0,5000,10000,20000,40000,np.inf]; labels=["<5k","5-10k","10-20k","20-40k",">40k"]
bid["band"]=pd.cut(bid["MIX_TN_num"], bins=edges, labels=labels, right=False)
binned = bid.groupby("band", observed=True).agg(
    n=("Contract_Id","size"),
    win_well=("is_win_well", lambda s: round(s.mean()*100,1)),
    bomb=("is_bomb", lambda s: round(s.mean()*100,1)),
    med_mix=("MIX_TN_num", lambda s: round(s.median()))).reset_index()
print(binned.to_string(index=False))
print("\n>>> win-well is flat to 40k then JUMPS to ~24% -> the edge is at ~40k, not 10k")

### (6) Is "large" state-relative?

In [ ]:
reg = bid.groupby("State")["MIX_TN_num"].agg(n="size", median="median",
      p90=lambda s: float(np.percentile(s,90))).round(0)
print(reg[reg["n"]>=10].sort_values("median", ascending=False).to_string())

### (7) Candidate thresholds vs the current 10k

In [ ]:
cands=[("Current 10k",CURRENT),("P75",pcts[75]),("P90",pcts[90]),
       ("1.5xIQR",iqr_fence),("K-means",kmeans_break),("GMM",gmm_cross)]
tab=pd.DataFrame([{ "candidate":nm, "threshold":round(float(t)),
   "pctile_bid":round(float((m<t).mean()*100)), "pct_bid_above":round(float((m>t).mean()*100),1)}
   for nm,t in cands if not (isinstance(t,float) and np.isnan(t))])
print(tab.to_string(index=False))

### (8) The figure — distribution + outcome shift

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(13,9))
ax[0,0].hist(m_all,bins=40,range=(0,80000),density=True,alpha=.35,color="#bdbdbd",label="market")
ax[0,0].hist(m,bins=40,range=(0,80000),density=True,alpha=.65,color="#2c7fb8",label="CSI bids")
for x,c in [(10000,"#d73027"),(kmeans_break,"#1a9850"),(pcts[90],"#7b3294")]: ax[0,0].axvline(x,color=c,ls="--",lw=2)
ax[0,0].set_title("(a) MIX_TN — bid vs market"); ax[0,0].legend(fontsize=8)
ax[0,1].hist(logm,bins=30,density=True,alpha=.55,color="#2c7fb8"); ax[0,1].axvline(np.log10(10000),color="#d73027",ls="--",lw=2)
ax[0,1].set_title("(b) log10(MIX_TN)")
xs=np.sort(m.values); ys=np.arange(1,len(xs)+1)/len(xs)
ax[1,0].plot(xs,ys,color="#2c7fb8",lw=2); ax[1,0].axvline(10000,color="#d73027",ls="--",lw=2); ax[1,0].set_xlim(0,80000)
ax[1,0].set_title(f"(c) ECDF — 10k = {(m<10000).mean()*100:.0f}th pctile")
bb=binned; xp=np.arange(len(bb))
ax[1,1].bar(xp-.2,bb["win_well"],.4,color="#1a9850",label="win-well %"); ax[1,1].bar(xp+.2,bb["bomb"],.4,color="#d73027",label="bomb %")
ax[1,1].set_xticks(xp); ax[1,1].set_xticklabels(bb["band"]); ax[1,1].axvline(1.5,color="#d73027",ls="--"); ax[1,1].legend(fontsize=8)
ax[1,1].set_title("(d) Outcome by band — shift is at >40k")
fig.suptitle("Where does Large Volume begin?", fontweight="bold"); fig.tight_layout(rect=[0,0,1,.97]); plt.show()

### (9) Light data check for the OTHER archetypes' cut-offs
Backs `2026-06-15_archetype_decision_rules_PROPOSED.md`: are `n_mat_types≥5`, `prop_GRADE≥0.50`, `prop_BASE+SHLDR≥0.60` natural lines?

In [ ]:
q = df[df["has_quantity"]==True]
print("n_mat_types (Complex uses >=5):"); print(q["n_mat_types"].value_counts().sort_index().to_string())
print("  share >=5:", round((q["n_mat_types"]>=5).mean()*100,1), "%  (>=4 would be", round((q["n_mat_types"]>=4).mean()*100,1),"%)")
g = q["prop_GRADE_CY"].dropna(); g=g[g>0]
print("prop_GRADE_CY where>0: P50/P75/P90 =", {p:round(float(np.percentile(g,p)),2) for p in [50,75,90]})
bs = q["prop_BASE_TN"].fillna(0)+q["prop_SHLDR_TN"].fillna(0)
print("base+shoulder >=0.60:", int((bs>=.6).sum()), " also mix<0.20:", int(((bs>=.6)&(q["prop_MIX_TN"].fillna(0)<.2)).sum()))
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
q["n_mat_types"].value_counts().sort_index().plot.bar(ax=ax[0],color="#7b3294"); ax[0].axvline(3.5,color="r",ls="--"); ax[0].set_title("n_mat_types (>=5=Complex)")
ax[1].hist(g,bins=20,color="#d7191c"); ax[1].axvline(.5,color="k",ls="--"); ax[1].set_title("prop_GRADE_CY (>=.50=Earthwork)")
ax[2].hist(bs[bs>0],bins=20,color="#abdda4"); ax[2].axvline(.6,color="k",ls="--"); ax[2].set_title("base+shoulder share (>=.60)")
fig.tight_layout(); plt.show()

## Takeaway
- **10k is the natural small/large seam** (GMM ≈ 9.2k) — keep it as the floor.
- But the **competitive edge is at ~40k** (win-well 11%→24%) → propose a **Major tier**.
- "Large" is **state-relative** (ND ~9k vs SD ~30k).
- Other cut-offs: `n_mat_types` tops out at 6 (weak complexity proxy); grade-share and base+shoulder lines are reasonable.

Full writeup: `2026-06-15_large_project_threshold_findings.md` · rules to test: `2026-06-15_archetype_decision_rules_PROPOSED.md`.